# Interactive Scorecard What-If

This notebook demonstrates the interactive what-if widget. Adjust any slider or dropdown and the score + waterfall chart update in real time.

In [ ]:
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from ScoreCardModel import BinningTransformer, WOETransformer
from ScoreCardModel.analytics.selection import rank_features, select_by_correlation

In [ ]:
# ── Load data with proper column names ──
COLUMN_MAP = {
    'x1': 'LIMIT_BAL', 'x2': 'SEX', 'x3': 'EDUCATION', 'x4': 'MARRIAGE', 'x5': 'AGE',
    'x6': 'PAY_0', 'x7': 'PAY_2', 'x8': 'PAY_3', 'x9': 'PAY_4', 'x10': 'PAY_5', 'x11': 'PAY_6',
    'x12': 'BILL_AMT1', 'x13': 'BILL_AMT2', 'x14': 'BILL_AMT3', 'x15': 'BILL_AMT4',
    'x16': 'BILL_AMT5', 'x17': 'BILL_AMT6',
    'x18': 'PAY_AMT1', 'x19': 'PAY_AMT2', 'x20': 'PAY_AMT3', 'x21': 'PAY_AMT4',
    'x22': 'PAY_AMT5', 'x23': 'PAY_AMT6',
}

data = fetch_openml('default-of-credit-card-clients', as_frame=True, parser='pandas', version=1)
X = data.data.rename(columns=COLUMN_MAP)
y = (data.target == '0').astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [ ]:
# ── Feature selection ──
ranking = rank_features(X_train, y_train, n_bins=5)
accept = ranking[ranking['Recommendation'] == 'Accept']['Feature'].tolist()
investigate = ranking[
    (ranking['Recommendation'] == 'Investigate') & (ranking['Monotonicity'] == 'decreasing')
]['Feature'].tolist()
final = select_by_correlation(X_train[accept + investigate], max_corr=0.7)
print(f"Selected features ({len(final)}): {final}")

In [ ]:
# ── Build and fit pipeline ──
pipe = Pipeline([
    ('binning', BinningTransformer(n_bins=5)),
    ('woe', WOETransformer(method='empirical_logit')),
    ('model', LogisticRegression(max_iter=1000)),
])
pipe.fit(X_train[final], y_train)

In [ ]:
# ── View the scorecard table ──
from ScoreCardModel.score_card.transformers import ScoreCardTransformer
lr, bt, wt = pipe.named_steps['model'], pipe.named_steps['binning'], pipe.named_steps['woe']
sct = ScoreCardTransformer(lr, bt, wt)
sct

In [ ]:
# ── Interactive what-if widget ──
# Install required: pip install scorecard-toolkit[interactive]
from ScoreCardModel.interactive import ScorecardWidget

widget = ScorecardWidget(pipe, X_train)
widget.display()

---

Adjust the controls above. The score and waterfall chart update automatically. Features with positive contribution (green) increase the score (lower risk), negative (red) decrease it.